In [1]:
%run common_imports.py

%matplotlib qt
%config InlineBackend.figure_format = 'retina'
sns.set_context("talk")

%reload_ext autoreload
%autoreload 2
pd.options.display.max_rows = 600
pd.set_option('display.float_format', lambda x: '%.9f' % x)

dj.config['display.limit'] = 10**3  

os.environ["SPYGLASS_USE_TRANSACTIONS"] = "1"  
os.environ['KACHERY_API_KEY'] = "RhysjLwgmBAt2ObCyXXaDnqAv2kTdYRa"

[2026-03-10 18:09:38,739][INFO]: DataJoint is configured from /media/labuser/NA_1_2025/spyglass/wilbur/dj_local_conf.json
[2026-03-10 18:09:39,221][INFO]: DataJoint 0.14.9 connected to anirudh@172.16.102.154:3306


In [2]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

### Load data

In [3]:
#extract position data
#read from csv
trialized_position = pd.read_csv("/media/labuser/NA_1_2025/spyglass/wilbur/analysis/position/trialized_position.csv", index_col = "time")

In [4]:
#extract spikes
#read from npz
data = np.load("/media/labuser/NA_1_2025/spyglass/wilbur/analysis/final_spikes/mfpc_spikes.npz", allow_pickle=True)
mpfc_spikes = [data[f"arr_{i}"] for i in range(len(data.files))]


### Prepare dataframe for regression

In [42]:
def fit_glm_all_units(formula: str,
                      cov_df: pd.DataFrame,
                      spike_counts_masked: np.array,
                      unit_ids: np.array,
                      bin_size = 0.002):
    
    rows = []
    for i, uid in enumerate(unit_ids):
        df = cov_df.copy()
        df["spike_count"] = spike_counts_masked[i]  # pre-masked counts
        try:
            res = smf.glm(formula, data=df, family=sm.families.Poisson()).fit(disp=False)
            rows.append(dict(                                                                                       
                unit=uid,   
                aic=res.aic,
                llf=res.llf,
                deviance=res.deviance,
                n_params=len(res.params),
                n_obs=int(res.nobs),
                converged=res.converged,
                coef=res.params.to_dict(),
                bse=res.bse.to_dict(),
                deviance_null = res.null_deviance,
                df_model = res.df_model
            ))

        except Exception as e:
            rows.append(dict(
                unit=uid, aic=np.nan, llf=np.nan, deviance=np.nan,
                n_params=np.nan, n_obs=np.nan, converged=False,
                coef=None, bse=None, deviance_null=np.nan,
                df_model=np.nan, error=str(e)          # ← add error
            ))


    return pd.DataFrame(rows)

In [6]:
BIN_SIZE = 0.002  

bin_edges = np.arange(
    trialized_position.index.min(), trialized_position.index.max() + BIN_SIZE, BIN_SIZE
)
bin_centers = bin_edges[:-1] + BIN_SIZE / 2

spike_counts = np.array([np.histogram(spikes, bins=bin_edges)[0] for spikes in mpfc_spikes]) 

In [7]:
def interp_col(col_values, times, bin_centers):
    if pd.api.types.is_numeric_dtype(col_values):
        return np.interp(bin_centers, times, col_values.astype(float))
    else:
        idx = np.searchsorted(times, bin_centers).clip(0, len(times) - 1)
        return col_values.iloc[idx].values

cols_to_interp = [c for c in trialized_position.columns if c != "video_frame_ind"]
times = trialized_position.index.astype(float).values

interpolated = {col: interp_col(trialized_position[col], times, bin_centers) for col in cols_to_interp}

interp_trialised_position = pd.DataFrame(interpolated, columns=cols_to_interp)
interp_trialised_position.insert(0, "time_bin_center", bin_centers)

mask = (interp_trialised_position["zone"]=="run") &\
    (interp_trialised_position["trial_type"].isin(["outbound", "inbound"]))


cov_df = interp_trialised_position[mask]
spike_counts_masked = spike_counts[:, mask]
unit_ids = np.arange(0, len(spike_counts_masked))

# print(cov_df.head(1))
# print(spike_counts_masked[0].shape)
#print(unit_ids)

In [54]:
cov_df = cov_df.rename(columns={"left/right": "choice"})

### Models:

#### Single variable models:
1. Null model (constant rate)
2. spike_count ~ trial_type (categorical)
3. spike_count ~ left/right choice (categorical)
4. spike_count ~ speed (linear)
5. spike_count ~ bs(speed, df = ) (spline)
6. spike_coun ~ bs(linear_position, df = ) (spline)

#### Mutli-variable models:

### Null model

#### Fit on one unit 

In [55]:
unit_idx = 9
spk_cov_df = cov_df.copy()
spk_cov_df["spike_count"] = spike_counts_masked[unit_idx]

In [9]:
model_constant = smf.glm("spike_count ~ 1", data=spk_cov_df, family=sm.families.Poisson())
results_constant = model_constant.fit()

print(results_constant.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1835243
Model:                            GLM   Df Residuals:                  1835242
Model Family:                 Poisson   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -15493.
Date:                Tue, 10 Mar 2026   Deviance:                       27031.
Time:                        18:09:56   Pearson chi2:                 1.83e+06
No. Iterations:                     8   Pseudo R-squ. (CS):              0.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -6.8328      0.022   -303.889      0.0

In [10]:
# Interpret the coefficient
mean_count_per_bin = np.exp(results_constant.params["Intercept"])
mean_rate_hz = mean_count_per_bin / BIN_SIZE

print(f"β₀ = {results_constant.params['Intercept']:.4f}")
print(f"exp(β₀) = {mean_count_per_bin:.4f} spikes/bin")
print(f"Firing rate = {mean_rate_hz:.2f} Hz")
print(f"Observed mean = {spk_cov_df['spike_count'].mean():.4f} spikes/bin")

β₀ = -6.8328
exp(β₀) = 0.0011 spikes/bin
Firing rate = 0.54 Hz
Observed mean = 0.0011 spikes/bin


#### Fit on all units

In [ ]:
# null_model_all = fit_glm_all_units("spike_count ~ 1", cov_df, spike_counts_masked, unit_ids)

/home/labuser/miniforge3/envs/spyglass/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)


In [43]:
# null_model_all.to_csv(f"{base_dir}/analysis/null_model_all.csv")
null_model_all = pd.read_csv(f"{base_dir}/analysis/null_model_all.csv", index_col=0)
null_model_all["model"] = "null"

In [44]:
null_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,model
0,0,146661.528243705,-73329.764121853,122310.026719650,1,1835243,True,{'Intercept': -5.014316233297876},{'Intercept': 0.009057287368239611},null
1,1,62875.359086194,-31436.679543097,53915.517969278,1,1835243,True,{'Intercept': -6.01508594098356},{'Intercept': 0.01493869039855228},null
2,2,27650.746202234,-13824.373101117,24176.746202234,1,1835243,True,{'Intercept': -6.963348560546998},{'Intercept': 0.024000746649013627},null
3,3,83401.915912946,-41699.957956473,70930.847384752,1,1835243,True,{'Intercept': -5.684272558606533},{'Intercept': 0.012661271297716607},null
4,4,115959.900830617,-57978.950415308,97535.900830617,1,1835243,True,{'Intercept': -5.294533754779503},{'Intercept': 0.010419493518636521},null


### Trial type

#### Fit on one unit

In [25]:
model_trial_type = smf.glm("spike_count ~ trial_type", data=spk_cov_df, family=sm.families.Poisson())
results_trial_type = model_trial_type.fit()

print(results_trial_type.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1835243
Model:                            GLM   Df Residuals:                  1835241
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -15273.
Date:                Tue, 10 Mar 2026   Deviance:                       26590.
Time:                        18:25:15   Pearson chi2:                 1.83e+06
No. Iterations:                     9   Pseudo R-squ. (CS):          0.0002400
Covariance Type:            nonrobust                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -6

In [31]:
rate_inbound  = np.exp(results_trial_type.params["Intercept"]) / BIN_SIZE        # Hz
rate_outbound = np.exp(results_trial_type.params["Intercept"] + results_trial_type.params["trial_type[T.outbound]"]) / BIN_SIZE
ratio         = np.exp(results_trial_type.params["trial_type[T.outbound]"])      # outbound/inbound rate ratio

print("inbound rate: ", rate_inbound)
print("outbound rate: ", rate_outbound)
print("outbound/inbound: ", ratio)

inbound rate:  0.7729987805818047
outbound rate:  0.2777832207690415
outbound/inbound:  0.3593579029451577


#### Fit all units 

In [ ]:
# trial_type_model_all = fit_glm_all_units("spike_count ~ trial_type", cov_df, spike_counts_masked, unit_ids)

/home/labuser/miniforge3/envs/spyglass/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)


In [48]:
# trial_type_model_all.to_csv(f"{base_dir}/analysis/trial_type_model_all.csv")
trial_type_model_all = pd.read_csv(f"{base_dir}/analysis/trial_type_model_all.csv", index_col=0)
trial_type_model_all["model"] = "trial_type"

In [49]:
trial_type_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,model
0,0,146354.172384433,-73175.086192217,122000.670860378,2,1835243,True,"{'Intercept': -4.874482451548208, 'trial_type[...","{'Intercept': 0.011631052629827653, 'trial_typ...",trial_type
1,1,62822.257539506,-31409.128769753,53860.416422589,2,1835243,True,"{'Intercept': -5.915530564583954, 'trial_type[...","{'Intercept': 0.01957400707490679, 'trial_type...",trial_type
2,2,27628.592982041,-13812.296491021,24152.592982041,2,1835243,True,"{'Intercept': -7.081904955371571, 'trial_type[...","{'Intercept': 0.03507153119208145, 'trial_type...",trial_type
3,3,83311.912599513,-41653.956299756,70838.844071318,2,1835243,True,"{'Intercept': -5.575779636466067, 'trial_type[...","{'Intercept': 0.016515957975578685, 'trial_typ...",trial_type
4,4,115866.653900175,-57931.326950087,97440.653900175,2,1835243,True,"{'Intercept': -5.202843731694705, 'trial_type[...","{'Intercept': 0.01370634840006244, 'trial_type...",trial_type


### Choice

#### Fit on one unit

In [63]:
choice_mask = spk_cov_df["trial_type"]=="outbound"
choice_spk_cov_df = spk_cov_df[choice_mask]
model_choice= smf.glm("spike_count ~ choice", data=choice_spk_cov_df, family=sm.families.Poisson())
results_choice = model_choice.fit()

print(results_choice.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               867583
Model:                            GLM   Df Residuals:                   867581
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4007.2
Date:                Tue, 10 Mar 2026   Deviance:                       7050.5
Time:                        19:05:41   Pearson chi2:                 8.67e+05
No. Iterations:                    10   Pseudo R-squ. (CS):          0.0002019
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -6.4212      0.076    -

In [67]:
rate_left  = np.exp(results_choice.params["Intercept"]) / BIN_SIZE        # Hz
rate_right = np.exp(results_choice.params["Intercept"] + results_choice.params["choice[T.right]"]) / BIN_SIZE
ratio         = np.exp(results_choice.params["choice[T.right]"])      # outbound/inbound rate ratio

print("left rate: ", rate_left)
print("right rate: ", rate_right)
print("right/left: ", ratio)

left rate:  0.8133174791934886
right rate:  0.203945659961309
right/left:  0.2507577485775272


In [ ]:
#TODO: compare this against a null model that is OUTBOUND only

#### Fit for all units

In [71]:
choice_model_all = fit_glm_all_units("spike_count ~ choice", cov_df[choice_mask], spike_counts_masked[:, choice_mask], unit_ids) #fit only on outbound trials

/home/labuser/miniforge3/envs/spyglass/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)


In [74]:
choice_model_all.to_csv(f"{base_dir}/analysis/choice_model_all.csv")
choice_model_all = pd.read_csv(f"{base_dir}/analysis/choice_model_all.csv", index_col=0)
choice_model_all["model"] = "choice"

In [75]:
choice_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,51945.143850145,-25970.571925072,42353.461616311,2.000000000,867583.000000000,True,"{'Intercept': -3.5083827877384812, 'choice[T.r...","{'Intercept': 0.017823067247957607, 'choice[T....",49891.960876831,1.000000000,NaN,choice
1,1,22912.949650976,-11454.474825488,19171.108534059,2.000000000,867583.000000000,True,"{'Intercept': -4.331618392947591, 'choice[T.ri...","{'Intercept': 0.026899608277649346, 'choice[T....",22981.346875516,1.000000000,NaN,choice
2,2,12235.800570111,-6115.900285055,10385.800570111,2.000000000,867583.000000000,True,"{'Intercept': -4.956255211077718, 'choice[T.ri...","{'Intercept': 0.03676073109408769, 'choice[T.r...",12637.415524655,1.000000000,NaN,choice
3,3,32304.974140526,-16150.487070263,27159.746729248,2.000000000,867583.000000000,True,"{'Intercept': -4.2988752543783235, 'choice[T.r...","{'Intercept': 0.026462806167679386, 'choice[T....",29948.910010403,1.000000000,NaN,choice
4,4,40408.210686358,-20202.105343179,32628.210686358,2.000000000,867583.000000000,True,"{'Intercept': -3.5211706857859935, 'choice[T.r...","{'Intercept': 0.017937400083207797, 'choice[T....",42051.179532311,1.000000000,NaN,choice
